In [1]:
import pandas as pd 
import requests
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

In [2]:
# carregando a chave da API do .env

load_dotenv('../.env')
API_KEY = os.getenv('OMDB_API_KEY')

In [ ]:
# carregando o csv enriquecido pelo tmdb

df = pd.read_csv('.../data/df_tmdb.csv')

In [ ]:
# novas coluna que queremos adicionar no nosso dataset

metascore_critica = []

In [ ]:
# limite porque a api limita a 1000 requisições diárias

limite_diario = 950
contador = 0

In [ ]:
# coleta de dados dos omdb

for index, row in tqdm(df.iterrows(), total=len(df)):
    
    if 'metascore_critica' in df.columns:
        nota_existente = row['metascore_critica']

        if pd.notna(nota_existente) and nota_existente != -1:
            metascore_critica.append(nota_existente)
            continue
        
    if contador >= limite_diario:
        metascore_critica.append(None)
        continue
        
    imdb_id = row['tconst']
    url_omdb = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_API_KEY}"
    
    try:
        response = requests.get(url_omdb)
        
        if response.status_code == 200:
            data = response.json()
            score = data.get('Metascore')
            
            if score and score != "N/A":
                metascore_critica.append(int(score))
            else:
                metascore_critica.append(-1)
        else:
            metascore_critica.append(None)
            
        contador += 1
        time.sleep(0.05)
        
    except Exception as e:
        metascore_critica.append(None)

In [ ]:
# adicionando as colunas no .csv

df['metascore_critica'] = metascore_critica

df.to_csv('.../data/df_tmdb.csv', index=False)